In [ ]:
import pickle
import numpy as np
from tqdm import tqdm

def load(filename):
    path = f"./data/cifar-10-batches-py/{filename}"
    with open(path, 'rb') as fo: return pickle.load(fo, encoding='bytes')

def load_cifar10():
    batches = [load(f"data_batch_{i}") for i in range(1, 5 + 1)]
    X_tr = np.concatenate([b[b"data"] for b in batches], axis=0)
    Y_tr = np.concatenate([b[b"labels"] for b in batches], axis=0)
    test = load("test_batch")
    X_te = np.array(test[b"data"])
    Y_te = np.array(test[b"labels"])
    return X_tr, Y_tr, X_te, Y_te

X_tr, Y_tr, X_te, Y_te = load_cifar10()

print(f"{X_tr.shape = }")
print(f"{Y_tr.shape = }")
print(f"{X_te.shape = }")
print(f"{Y_te.shape = }")

In [3]:
class NearestNeighbor:
    METRICS = {"l1": (np.int16, np.abs), "l2": (np.int32, np.square)}

    def __init__(self, metric = "l1"): self.dtype, self.fn = self.METRICS[metric]

    def train(self, X, y): self.X, self.y = X.astype(self.dtype), y

    def predict(self, X, batch_size=1, progress=False):
        n_pred = X.shape[0]
        y = np.zeros(n_pred, dtype=self.y.dtype)
        for start in tqdm(range(0, n_pred, batch_size), desc="Predicting", disable=not progress):
            end = min(start+batch_size, n_pred)
            y[start:end] = self.y[self.fn(self.X[None, :, :] - X[start:end, None, :]).sum(axis=2).argmin(axis=1)]

        return y

In [4]:
n_test = 100

In [5]:
nn = NearestNeighbor(metric="l2")
nn.train(X_tr, Y_tr)
Y_pred = nn.predict(X_te[:n_test], batch_size=1, progress=True)
acc = np.mean(Y_pred == Y_te[:n_test])
print(f"acc = {acc:.4f}")

Predicting: 100%|██████████| 100/100 [00:08<00:00, 12.19it/s]

acc = 0.3100


In [ ]:
nn = NearestNeighbor(metric="l2")
nn.train(X_tr, Y_tr)
Y_pred = nn.predict(X_te, progress=True)
acc = np.mean(Y_pred == Y_te)
print(f"acc = {acc:.4f}")

Predicting:  29%|██▉       | 2896/10000 [03:59<09:44, 12.15it/s]